# Decoded quantum interferometry for Max-XORSAT

All 15 source stages, the 15-bit unary example, both six-bit Dicke states, the six-cycle Max-XORSAT graph and all 22 lookup entries. Use 2,048 shots for each preparation and 10,000 DQI samples.

Prepare the unary example from its known classical count and use a reversible preparation specialized to the specified unary basis input for the second Dicke example. Regenerate the incidence and decoder tables and solve the original three-by-three eigenproblem. The native circuit keeps both output words and tests the zero-error word directly from measured results.

## Run the native unary and Dicke preparations

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.algorithms.search_and_optimization.dqi.dqi_max_xorsat.overview.model import (
    DQI_SHOTS,
    EDGES,
    NODES,
    majority_errors,
)
from examples.synthesis.algorithms.search_and_optimization.dqi.dqi_max_xorsat.overview.plots import (
    save_graph,
    save_scores,
)
from examples.synthesis.algorithms.search_and_optimization.dqi.dqi_max_xorsat.overview.workflow import (
    run_workflow,
)

folder = Path("artifacts/development/dqi-workflow")
result = run_workflow()
print("Unary observation:", result.unary)


## Read the six-bit weight-four Dicke sample

In [ ]:
assert all(word.bit_count() == 4 for word, _ in result.dicke_four)


## Read the unary-to-Dicke sample

In [ ]:
assert all(word.bit_count() == 2 for word, _ in result.dicke_two)
print("Dicke support sizes:", len(result.dicke_four), len(result.dicke_two))


## Inspect the six-column parity encoder

In [ ]:
print("GF(2) columns:", result.model.columns)


## Inspect the coherent decoder sequence

In [ ]:
print(
    "Prepare symmetric weights, phase the clauses, encode the syndrome and uncompute the error word."
)


## Generate the source graph

In [ ]:
print("Source graph:", NODES, EDGES)
display(Image(filename=str(save_graph(folder / "graph.png"))))


## Enumerate all assignment scores

In [ ]:
print("Score support:", sorted(set(result.model.objective)))


## Enumerate the weight-zero through weight-two errors

In [ ]:
assert len(result.model.syndrome_table) == 22


## Inspect the actual decoder entries

In [ ]:
print("Syndrome lookup:", result.model.syndrome_table)


## Compare the illustrative majority rule

In [ ]:
print(
    "Decoder entries and majority misses:",
    len(result.model.syndrome_table),
    majority_errors(result.model),
)


## Regenerate the principal weight eigenvector

In [ ]:
print("Optimal weight amplitudes:", result.model.weights)


## Inspect the original-size native program

In [ ]:
print("Native resources:", result.program.report["resources"])


## Read the actual finite sample

In [ ]:
print("DQI shots:", sum(count for _, count in result.dqi))
assert sum(count for _, count in result.dqi) == DQI_SHOTS


## Verify that the measured decoder cleared the error

In [ ]:
assert all(error == 0 for (error, _), _ in result.dqi)


## Compare score distributions and expectations

In [ ]:
print("Sampled DQI score:", result.mean_score)
print("Exact uniform score:", sum(result.model.objective) / len(result.model.objective))
display(Image(filename=str(save_scores(result, folder / "scores.png"))))
